# KArSL Inventory Verification

Read-only verification of packaged metadata and provenance. No original video archives or downloads are required. Run after unzipping the package; set SIGNOVA_KARSL_ROOT if running elsewhere.


In [ ]:
from pathlib import Path
import os
import pandas as pd
root = Path(os.environ.get("SIGNOVA_KARSL_ROOT", ".")).expanduser().resolve()
if not (root / "data/metadata/karsl_metadata.csv").is_file():
    candidate = Path.cwd() / "karsl"
    if (candidate / "data/metadata/karsl_metadata.csv").is_file():
        root = candidate.resolve()
    else:
        raise FileNotFoundError("Set SIGNOVA_KARSL_ROOT to the unpacked karsl folder")
print("KArSL root:", root)


In [ ]:
m = pd.read_csv(root / "data/metadata/karsl_metadata.csv", low_memory=False)
sources = pd.read_csv(root / "data/metadata/dataset_sources.csv", low_memory=False)
access = pd.read_csv(root / "data/metadata/karsl_video_access_manifest.csv", low_memory=False)
checks = {
    "Metadata rows": len(m) == 75514,
    "502 original classes": m["class_label"].nunique() == 502,
    "Three signers": m["signer_id"].nunique() == 3,
    "Thirty archives": m["archive_relpath"].nunique() == 30,
    "Unique sample IDs": m["sample_id"].is_unique,
    "Video-access references": len(access) == 75514,
    "Source register present": len(sources) >= 1
}
for name, ok in checks.items():
    print(name, "PASS" if ok else "FAIL")
assert all(checks.values())
print("Video archive contents individually reopened: NO")
